# fastapi async + redis cache for an inference endpoint

putting redis in front of an inference endpoint to cache idempotent requests. using `aioredis` (now `redis.asyncio`).

In [ ]:
code = '''
import hashlib, json, os
import joblib
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel
import redis.asyncio as redis

app = FastAPI()
model = joblib.load("iris.joblib")
r = redis.from_url(os.getenv("REDIS_URL", "redis://localhost:6379/0"))

class Req(BaseModel):
    features: list[float]

def key(features):
    return "pred:" + hashlib.sha1(json.dumps(features).encode()).hexdigest()

@app.post("/predict")
async def predict(req: Req):
    k = key(req.features)
    hit = await r.get(k)
    if hit:
        return {"pred": int(hit), "cached": True}
    pred = int(model.predict(np.array([req.features]))[0])
    await r.setex(k, 3600, pred)
    return {"pred": pred, "cached": False}
'''
print(code)


## things i learned
- aioredis 2.x is now `redis.asyncio`, the import path moved this year. lots of stale stackoverflow.
- never cache when the model output is non-deterministic (e.g. with dropout at inference).
- ttl on the cache should match how often the model itself changes, not the request rate.

found a corner case in the input pipeline. fixed.

added gradient clipping at 1.0, training stabilized.

## obs
weird: f1 dropped when batch size went up. need to look.

## todo
plot loss curves.

In [ ]:
# small batch
BATCH = 4

## obs
lr=3e-5 way better than 5e-5 here.

In [ ]:
for i in range(3):
    print(i)